In [6]:
from datetime import datetime
import sys
from time import sleep
from warnings import warn
import traceback

In [22]:
class TaskSuccess:
    def __init__(self):
        self.success: bool = False


class Task:
    def __init__(self, task:str, stdout_path:str, verbose:bool=False):
        self.task: str = task
        self.stdout_path: str = stdout_path
        self.verbose: bool = verbose
        self.task_success = TaskSuccess()
        
        self._start: datetime | None = None
        self._end: datetime | None = None
        self._original_stdout = sys.stdout
        self._original_stderr = sys.stderr
        self._separator: str = "-"*70
        
    def __enter__(self):
        f = open(self.stdout_path, mode="a")
        sys.stdout = f
        sys.stderr = f
        
        self._start = datetime.now()
        print(f"\tStarted task '{self.task}' on {self._start}")
        print(self._separator, end="\n\n")
        return self.task_success

    def __exit__(self, err, value, traceback):
        self._end = datetime.now()
        length_in_sec = (self._end - self._start).total_seconds()
        m, s = divmod(length_in_sec, 60)
        
        print("\n" + self._separator)
        if err:
            self.task_success.success = False
            print(f"\tThe task failed after {m:.0f} minutes and {s:.1f} seconds on {self._end.isoformat(sep=' ')}")
            print(f"\tReason: {err.__name__} - {value}", end="\n"*3)
        else:
            self.task_success.success = True
            print(
                f"\tSuccessfully ended task on {self._end.isoformat(sep=' ')}." 
                f"It took {m:.0f} minutes and {s:.1f} seconds", 
                end="\n"*3
            )
    
        sys.stdout = self._original_stdout
        sys.stderr = self._original_stderr
        return True

In [ ]:
with Task("STT", "logs.txt") as task_success:
    print("doing stuff...")
    sleep(3)
    warn("Some warning")
    # raise ValueError("Error message")
print(f"Exited task. Success: {task_success.success}")